# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook provides a structured workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is defined by a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Access title and description
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Here, we list all record sets available in the dataset, along with their fields and columns, referencing each by their `@id`.

In [ ]:
# List all record sets with their Fields and Columns (@id references)
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    record_set_id = rs['@id'] if '@id' in rs else 'unknown'
    print(f"- RecordSet @id: {record_set_id}")
    # List fields
    for field in rs.get('field', []):
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
        print(f"  - Field @id: {field_id}")
    # List columns
    for col in rs.get('column', []):
        col_id = col['@id'] if isinstance(col, dict) and '@id' in col else str(col)
        print(f"  - Column @id: {col_id}")

## 3. Data Extraction
Load data from each RecordSet into a DataFrame for analysis. All references use `@id`. Below, we iterate over available record sets, load their records, and create pandas DataFrames for each.


In [ ]:
# Extract data from each RecordSet identified by its @id
dataframes = {}
record_set_ids = []

# Build list of @ids for record sets
for rs in dataset.record_sets:
    record_set_ids.append(rs['@id'])
    
for rs_id in record_set_ids:
    # Load all records from the record set by @id
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"RecordSet @id: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(), "\n")
    else:
        print(f"RecordSet @id {rs_id} contains no records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, categorizing data, removing outliers, transforming distributions, and grouping by key attributes.

For demonstration, we'll select the first available numeric column in the first RecordSet and group by a categorical field if available. All operations reference columns and fields by their `@id`.

In [ ]:
# Identify the first populated RecordSet and its columns
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    print(f"Using RecordSet @id: {first_rs_id}")

    # Find numeric fields by dtype
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field @id: {numeric_field_id}")

        # Filter records for values above a threshold (e.g., mean)
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Find a groupable categorical field
        cat_cols = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id]
        if cat_cols:
            group_field_id = cat_cols[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count']).reset_index()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No categorical field available for grouping.")
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No dataframes extracted. Please check the dataset record sets.")

## 5. Visualization
Visualize data distributions and relationships.

Below we plot the distribution of the selected numeric field and optionally relationships with categorical groups, using fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if EDA identified valid columns
if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped boxplot if group field available
    if 'group_field_id' in locals():
        plt.figure(figsize=(10,4))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load, overview, and analyze the FAIR² Rangeland dataset using the `mlcroissant` library, referencing all entities by their `@id`. You can extend this workflow to other record sets, fields, or columns as appropriate and apply further statistical or ML analysis on the processed DataFrames.

Key steps:
- Accessed and loaded dataset metadata and records using Croissant schema.
- Provided overview of all available record sets and entities by `@id`.
- Extracted and processed data, demonstrating filtering, normalization, and grouping.
- Visualized numeric distributions and group relationships.

For further exploration, refer to the dataset schema for more fields, or expand the notebook with advanced processing and visualization techniques.